# 🏭 Ingesta GitHub → Microsoft Fabric Lakehouse
**Sesión 5 — Analítica de Datos con Power BI y SQL**  
**Instructor:** Walter Calcagno · Microsoft MVP Data Platform  

---
### ¿Qué hace este notebook?

| Paso | Detalle |
|------|---------|
| **1** | Lee los CSV desde el repositorio público en GitHub |
| **2** | `ventas_tienda.csv` → tabla `ventas_tienda` (Gold directo, Ejercicio Newbie) |
| **3** | `orders_historico.csv` → **Medallion Architecture** Bronze → Silver → Gold |
| **4** | Verifica las tablas creadas en el Lakehouse |

> **Pre-requisito:** Tener un Lakehouse adjunto a este notebook.  
> Si no: *New Lakehouse → crear → adjuntar al notebook.*


## 1️⃣  URLs — Repositorio GitHub

In [1]:
# URLs públicas del repositorio
BASE = "https://raw.githubusercontent.com/wcalcagno/Clases-AFP-Capital/refs/heads/main"

URL_VENTAS = f"{BASE}/ventas_tienda.csv"
URL_ORDERS = f"{BASE}/orders_historico.csv"

print("📦 Fuentes:")
print(f"  ventas_tienda    → {URL_VENTAS}")
print(f"  orders_historico → {URL_ORDERS}")


StatementMeta(, 7bb7d9bd-9719-48da-866c-2dc871389396, 3, Finished, Available, Finished, False)

📦 Fuentes:
  ventas_tienda    → https://raw.githubusercontent.com/wcalcagno/Clases-AFP-Capital/refs/heads/main/ventas_tienda.csv
  orders_historico → https://raw.githubusercontent.com/wcalcagno/Clases-AFP-Capital/refs/heads/main/orders_historico.csv


## 2️⃣  Importaciones

In [2]:
import pandas as pd
import requests
from io import StringIO
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType, DecimalType

print(f"✅ Imports OK  |  Spark {spark.version}")


StatementMeta(, 7bb7d9bd-9719-48da-866c-2dc871389396, 4, Finished, Available, Finished, False)

✅ Imports OK  |  Spark 3.5.5.5.4.20260403.6


## 3️⃣  Helper: lectura de CSV desde URL

In [3]:
def read_csv_url(url: str, label: str = "") -> "DataFrame":
    """Lee un CSV público → Spark DataFrame (via pandas)."""
    nombre = label or url.split("/")[-1]
    print(f"⬇️  Descargando {nombre} ...", end=" ")
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    pdf = pd.read_csv(StringIO(resp.text))
    print(f"OK  ({len(pdf):,} filas × {len(pdf.columns)} cols)")
    return spark.createDataFrame(pdf)

# Verificar conectividad
r = requests.head(URL_VENTAS, timeout=10)
print(f"✅ GitHub accesible (HTTP {r.status_code})")


StatementMeta(, 7bb7d9bd-9719-48da-866c-2dc871389396, 5, Finished, Available, Finished, False)

✅ GitHub accesible (HTTP 200)


---
## 🟢  Ejercicio Newbie — `ventas_tienda.csv`
Pipeline directo: **GitHub → limpieza → tabla Gold** lista para Power BI.

In [4]:
df_ventas_raw = read_csv_url(URL_VENTAS, "ventas_tienda.csv")
df_ventas_raw.printSchema()
df_ventas_raw.show(5, truncate=False)


StatementMeta(, 7bb7d9bd-9719-48da-866c-2dc871389396, 6, Finished, Available, Finished, False)

⬇️  Descargando ventas_tienda.csv ... OK  (100 filas × 5 cols)
root
 |-- Fecha: string (nullable = true)
 |-- Producto: string (nullable = true)
 |-- Categoria: string (nullable = true)
 |-- Cantidad: long (nullable = true)
 |-- Monto: long (nullable = true)

+----------+-----------------------+--------------+--------+-------+
|Fecha     |Producto               |Categoria     |Cantidad|Monto  |
+----------+-----------------------+--------------+--------+-------+
|2025-11-07|Hub USB-C 7 en 1       |Periféricos   |4       |147840 |
|2025-11-08|Switch 8 puertos D-Link|Redes         |2       |60620  |
|2025-11-09|Notebook HP 15s        |Computación   |2       |1098920|
|2025-11-13|Headset Gamer HyperX   |Audio         |1       |83600  |
|2025-11-14|SSD Samsung 500GB      |Almacenamiento|2       |117180 |
+----------+-----------------------+--------------+--------+-------+
only showing top 5 rows



In [5]:
df_ventas = (
    df_ventas_raw
    .withColumn("Fecha",     F.to_date("Fecha", "yyyy-MM-dd"))
    .withColumn("Cantidad",  F.col("Cantidad").cast(IntegerType()))
    .withColumn("Monto",     F.col("Monto").cast(LongType()))
    .withColumn("Categoria", F.trim(F.col("Categoria")))
    .withColumn("Producto",  F.trim(F.col("Producto")))
    .filter(F.col("Fecha").isNotNull())
    .filter(F.col("Monto") > 0)
)

print(f"✅ Filas limpias: {df_ventas.count():,}")
df_ventas.show(5, truncate=False)


StatementMeta(, 7bb7d9bd-9719-48da-866c-2dc871389396, 7, Finished, Available, Finished, False)

✅ Filas limpias: 100
+----------+-----------------------+--------------+--------+-------+
|Fecha     |Producto               |Categoria     |Cantidad|Monto  |
+----------+-----------------------+--------------+--------+-------+
|2025-11-07|Hub USB-C 7 en 1       |Periféricos   |4       |147840 |
|2025-11-08|Switch 8 puertos D-Link|Redes         |2       |60620  |
|2025-11-09|Notebook HP 15s        |Computación   |2       |1098920|
|2025-11-13|Headset Gamer HyperX   |Audio         |1       |83600  |
|2025-11-14|SSD Samsung 500GB      |Almacenamiento|2       |117180 |
+----------+-----------------------+--------------+--------+-------+
only showing top 5 rows



In [6]:
(df_ventas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ventas_tienda"))

print(f"✅ Tabla 'ventas_tienda' → {spark.table('ventas_tienda').count():,} registros")
print("   👉 Conectar Power BI: Lakehouse → SQL Endpoint → ventas_tienda")


StatementMeta(, 7bb7d9bd-9719-48da-866c-2dc871389396, 8, Finished, Available, Finished, False)

✅ Tabla 'ventas_tienda' → 100 registros
   👉 Conectar Power BI: Lakehouse → SQL Endpoint → ventas_tienda


---
## 🔵  Ejercicio Advance — Medallion Architecture

```
GitHub CSV  →  🥉 BRONZE (raw)  →  🥈 SILVER (limpio)  →  🥇 GOLD (dimensional)
```


### 🥉 BRONZE — Ingesta raw sin transformaciones

In [7]:
df_orders_raw = read_csv_url(URL_ORDERS, "orders_historico.csv")

df_bronze = (
    df_orders_raw
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source",      F.lit("github/orders_historico.csv"))
)

(df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("bronze_orders_raw"))

print(f"✅ BRONZE 'bronze_orders_raw' → {spark.table('bronze_orders_raw').count():,} registros")


StatementMeta(, 7bb7d9bd-9719-48da-866c-2dc871389396, 9, Finished, Available, Finished, False)

⬇️  Descargando orders_historico.csv ... OK  (5,633 filas × 7 cols)
✅ BRONZE 'bronze_orders_raw' → 5,633 registros


### 🥈 SILVER — Limpieza y estandarización

In [8]:
df_silver = spark.sql("""
    SELECT
        CAST(OrderID   AS INT)             AS OrderID,
        CAST(OrderDate AS DATE)            AS OrderDate,
        UPPER(TRIM(CustomerID))            AS CustomerID,
        UPPER(TRIM(ProductName))           AS ProductName,
        UPPER(TRIM(Category))              AS Category,
        CAST(Quantity  AS INT)             AS Quantity,
        CAST(UnitPrice AS DECIMAL(12,2))   AS UnitPrice,
        CAST(Quantity AS INT)
          * CAST(UnitPrice AS DECIMAL(12,2)) AS TotalAmount
    FROM bronze_orders_raw
    WHERE Quantity > 0
      AND UnitPrice > 0
      AND OrderDate IS NOT NULL
""")

(df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver_orders_clean"))

print(f"✅ SILVER 'silver_orders_clean' → {spark.table('silver_orders_clean').count():,} registros")
df_silver.show(5, truncate=False)


StatementMeta(, 7bb7d9bd-9719-48da-866c-2dc871389396, 10, Finished, Available, Finished, False)

✅ SILVER 'silver_orders_clean' → 5,633 registros
+-------+----------+----------+---------------------+-----------+--------+---------+-----------+
|OrderID|OrderDate |CustomerID|ProductName          |Category   |Quantity|UnitPrice|TotalAmount|
+-------+----------+----------+---------------------+-----------+--------+---------+-----------+
|11409  |2024-07-12|CLI-0214  |AUTOCAD LT 2025      |SOFTWARE   |3       |605990.00|1817970.00 |
|11410  |2024-07-15|CLI-0274  |WEBCAM LOGITECH C925E|PERIFERICOS|5       |98080.00 |490400.00  |
|11411  |2024-07-15|CLI-0217  |NOTEBOOK HP 15S      |LAPTOPS    |2       |568310.00|1136620.00 |
|11412  |2024-07-15|CLI-0013  |AIRPODS PRO 2        |AUDIO      |1       |277710.00|277710.00  |
|11413  |2024-07-15|CLI-0029  |MONITOR CURVO 32" QHD|MONITORES  |1       |495270.00|495270.00  |
+-------+----------+----------+---------------------+-----------+--------+---------+-----------+
only showing top 5 rows



### 🥇 GOLD — Modelo dimensional para Power BI

In [9]:
# Fact table
spark.sql("""
    CREATE OR REPLACE TABLE gold_fact_sales AS
    SELECT
        OrderID,
        OrderDate,
        CustomerID,
        ProductName,
        Category,
        Quantity,
        UnitPrice,
        TotalAmount,
        YEAR(OrderDate)                   AS SalesYear,
        MONTH(OrderDate)                  AS SalesMonth,
        QUARTER(OrderDate)                AS SalesQuarter,
        DATE_FORMAT(OrderDate, 'yyyy-MM') AS YearMonth
    FROM silver_orders_clean
    ORDER BY OrderDate
""")

print(f"✅ GOLD 'gold_fact_sales' → {spark.table('gold_fact_sales').count():,} registros")


StatementMeta(, 7bb7d9bd-9719-48da-866c-2dc871389396, 11, Finished, Available, Finished, False)

✅ GOLD 'gold_fact_sales' → 5,633 registros


In [10]:
# Dimension producto
spark.sql("""
    CREATE OR REPLACE TABLE gold_dim_producto AS
    SELECT DISTINCT
        ProductName,
        Category,
        ROUND(AVG(UnitPrice) OVER (PARTITION BY ProductName), 0) AS AvgUnitPrice
    FROM silver_orders_clean
""")

# Dimension tiempo
spark.sql("""
    CREATE OR REPLACE TABLE gold_dim_tiempo AS
    SELECT DISTINCT
        OrderDate                          AS Fecha,
        YEAR(OrderDate)                    AS Anio,
        MONTH(OrderDate)                   AS Mes,
        QUARTER(OrderDate)                 AS Trimestre,
        DATE_FORMAT(OrderDate, 'yyyy-MM')  AS AnioMes,
        CASE WHEN MONTH(OrderDate) <= 3 THEN 'Q1'
             WHEN MONTH(OrderDate) <= 6 THEN 'Q2'
             WHEN MONTH(OrderDate) <= 9 THEN 'Q3'
             ELSE 'Q4' END                AS QuarterLabel
    FROM silver_orders_clean
    ORDER BY Fecha
""")

print(f"✅ Dim Producto → {spark.table('gold_dim_producto').count():,} productos únicos")
print(f"✅ Dim Tiempo   → {spark.table('gold_dim_tiempo').count():,} fechas únicas")


StatementMeta(, 7bb7d9bd-9719-48da-866c-2dc871389396, 12, Finished, Available, Finished, False)

✅ Dim Producto → 45 productos únicos
✅ Dim Tiempo   → 522 fechas únicas


## 4️⃣  Verificación — Resumen de tablas

In [11]:
tables = [
    ("ventas_tienda",       "Newbie  — Gold directo"),
    ("bronze_orders_raw",   "Advance — Bronze"),
    ("silver_orders_clean", "Advance — Silver"),
    ("gold_fact_sales",     "Advance — Gold Fact"),
    ("gold_dim_producto",   "Advance — Gold Dim Producto"),
    ("gold_dim_tiempo",     "Advance — Gold Dim Tiempo"),
]

print("=" * 62)
print(f"  {'TABLA':<26} {'CAPA':<28} {'FILAS':>6}")
print("=" * 62)
for t, label in tables:
    try:
        n = spark.table(t).count()
        print(f"  {t:<26} {label:<28} {n:>6,}")
    except Exception as e:
        print(f"  {t:<26} ⚠️  {str(e)[:30]}")
print("=" * 62)
print("\n✅ Lakehouse listo → conectar Power BI via SQL Endpoint")


StatementMeta(, 7bb7d9bd-9719-48da-866c-2dc871389396, 13, Finished, Available, Finished, False)

  TABLA                      CAPA                          FILAS
  ventas_tienda              Newbie  — Gold directo          100
  bronze_orders_raw          Advance — Bronze              5,633
  silver_orders_clean        Advance — Silver              5,633
  gold_fact_sales            Advance — Gold Fact           5,633
  gold_dim_producto          Advance — Gold Dim Producto      45
  gold_dim_tiempo            Advance — Gold Dim Tiempo       522

✅ Lakehouse listo → conectar Power BI via SQL Endpoint


In [12]:
# Vista previa Gold para validar
print("🔍 Revenue por Categoría y Año:")
spark.sql("""
    SELECT
        Category,
        SalesYear,
        COUNT(*)                       AS Transacciones,
        SUM(Quantity)                  AS Unidades,
        ROUND(SUM(TotalAmount)/1e6, 2) AS Revenue_M_CLP
    FROM gold_fact_sales
    GROUP BY Category, SalesYear
    ORDER BY SalesYear, Revenue_M_CLP DESC
""").show(20, truncate=False)


StatementMeta(, 7bb7d9bd-9719-48da-866c-2dc871389396, 14, Finished, Available, Finished, False)

🔍 Revenue por Categoría y Año:
+--------------+---------+-------------+--------+-------------+
|Category      |SalesYear|Transacciones|Unidades|Revenue_M_CLP|
+--------------+---------+-------------+--------+-------------+
|LAPTOPS       |2024     |378          |1084    |740.41       |
|DESKTOPS      |2024     |261          |611     |480.91       |
|MONITORES     |2024     |291          |802     |285.46       |
|SOFTWARE      |2024     |315          |912     |190.83       |
|AUDIO         |2024     |381          |988     |172.58       |
|REDES         |2024     |334          |818     |166.81       |
|ALMACENAMIENTO|2024     |384          |1066    |156.49       |
|PERIFERICOS   |2024     |497          |1372    |137.87       |
|LAPTOPS       |2025     |368          |976     |685.56       |
|DESKTOPS      |2025     |270          |751     |677.13       |
|MONITORES     |2025     |326          |871     |334.28       |
|AUDIO         |2025     |365          |980     |177.66       |
|REDES   

---
## ✅ Notebook completado

| Tabla | Uso en Power BI |
|-------|-----------------|
| `ventas_tienda` | Ejercicio Newbie — conectar directo |
| `gold_fact_sales` | Tabla de hechos principal |
| `gold_dim_producto` | Dimensión producto |
| `gold_dim_tiempo` | Dimensión tiempo (drill-down Año → Mes) |

**Medida DAX — Ventas YoY %:**
```dax
Ventas YoY % =
VAR VentasActuales     = [Total Ventas]
VAR VentasAnioAnterior = CALCULATE([Total Ventas],
                           DATEADD(gold_dim_tiempo[Fecha], -1, YEAR))
RETURN DIVIDE(VentasActuales - VentasAnioAnterior, VentasAnioAnterior, BLANK())
```

> Si el notebook falla por acceso HTTP restringido en tu capacidad,
> usar **Pipeline → Copy Data → HTTP connector** con las mismas URLs.
